# 05 · Captura web para el Bloque 2 (crudo)

**Componente B · Company Profile, Bloque 2** · Capa: **bronce** (`crudo/web/`)

Captura, para cada empresa con web verificada, la **portada** y hasta **3 páginas institucionales** (nosotros, servicios, productos), y registra los **PDF oficiales** enlazados (memorias, informes, catálogos). No extrae todavía el tagline ni la descripción: eso es plata (notebook 06).

| Salida | Contenido |
| --- | --- |
| `crudo/web/<expediente>_<sha256>.html` | HTML crudo, inmutable (no se versiona en git) |
| `crudo/web/manifest.jsonl` | Una línea por página: URL, tipo, estado, fecha, hash, metadatos y PDF candidatos |

Mismas reglas éticas que el notebook 04. Solo texto institucional (LOPDP).

In [1]:
from pathlib import Path

ETL = Path.cwd().resolve()
if ETL.name == "notebooks":
    ETL = ETL.parent
DATOS = ETL / "datos"
CRUDO = DATOS / "crudo"      # bronce: datos tal como llegan, sin modificar
SCVS = CRUDO / "2025"        # descargas oficiales de la SCVS
WEB = CRUDO / "web"          # páginas capturadas de las webs oficiales
PLATA = DATOS / "plata"      # datos limpios y validados
ORO = DATOS / "oro"          # datos listos para publicar
for carpeta in (WEB, PLATA, ORO):
    carpeta.mkdir(parents=True, exist_ok=True)

In [2]:
import datetime as dt
import hashlib
import json
import logging
import re
import time
from urllib.parse import urlparse

import extruct
import pandas as pd
from scrapling.fetchers import FetcherSession
from scrapling.spiders import Request, Spider

USER_AGENT = "ObservatorioEPN-Bot/0.1 (+https://github.com/alda244/observatorio-epn; proyecto academico EPN)"
PAGINAS_EXTRA = 3
INSTITUCIONAL = re.compile(r"nosotros|quienes|qui[eé]nes|about|empresa|historia|servicio|producto|soluci|catalogo|cat[aá]logo|portafolio", re.I)
PDF_OFICIAL = re.compile(r"memoria|informe|sostenibilidad|reporte|catalogo|cat[aá]logo|brochure|folleto|portafolio", re.I)
MANIFEST = WEB / "manifest.jsonl"

## 1. Empresas con web verificada

In [3]:
urls = pd.read_csv(PLATA / "urls_piloto.csv", dtype={"expediente": int})
capturadas = set()
if MANIFEST.exists():
    capturadas = {json.loads(l)["expediente"] for l in MANIFEST.read_text(encoding="utf-8").splitlines() if l}
objetivo = urls[urls["url_verificada"].isin(["si", "manual"]) & ~urls["expediente"].isin(capturadas)]
objetivo[["expediente", "nombre", "url_oficial"]]

,expediente,nombre,url_oficial


## 2. Spider de captura

In [4]:
def tipo_pagina(url: str) -> str:
    ruta = urlparse(url).path.lower()
    if ruta in ("", "/"):
        return "portada"
    if re.search(r"nosotros|quienes|about|empresa|historia", ruta):
        return "nosotros"
    if re.search(r"servicio|producto|soluci|catalogo|portafolio", ruta):
        return "servicios"
    return "otra"

def metadatos(html: str, url: str) -> dict:
    datos = extruct.extract(html, base_url=url, syntaxes=["json-ld", "opengraph"], uniform=True)
    og = (datos.get("opengraph") or [{}])[0]
    org = next((d for d in datos.get("json-ld", []) if str(d.get("@type", "")).endswith("Organization")), {})
    return {
        "og_title": og.get("og:title"), "og_description": og.get("og:description"),
        "og_site_name": og.get("og:site_name"), "og_image": og.get("og:image"),
        "jsonld_organizacion": {k: org.get(k) for k in ("name", "description", "logo", "url", "sameAs") if org.get(k)},
    }


class CapturaBloque2(Spider):
    name = "captura_bloque2"
    robots_txt_obey = True
    concurrent_requests = 4
    concurrent_requests_per_domain = 1
    download_delay = 3.0
    max_blocked_retries = 0
    logging_level = logging.WARNING

    def __init__(self, objetivo: pd.DataFrame):
        self.objetivo = objetivo
        self.extra_por_empresa: dict[int, int] = {}
        super().__init__()

    def configure_sessions(self, manager):
        manager.add("etico", FetcherSession(impersonate=None, stealthy_headers=False,
                                            headers={"User-Agent": USER_AGENT}, retries=1, timeout=20))

    async def start_requests(self):
        for e in self.objetivo.itertuples():
            yield Request(e.url_oficial, sid="etico", callback=self.parse,
                          meta={"expediente": int(e.expediente), "dominio": urlparse(e.url_oficial).netloc})

    async def parse(self, r):
        exp, dominio = r.meta["expediente"], r.meta["dominio"]
        html = r.body.decode(r.encoding or "utf-8", errors="replace")
        enlaces = [(a.attrib.get("href", ""), " ".join(a.css("::text").getall())) for a in r.css("a[href]")]
        pdfs = sorted({r.urljoin(h) for h, t in enlaces if h.lower().endswith(".pdf") and PDF_OFICIAL.search(h + " " + t)})
        yield {
            "expediente": exp, "url": r.url, "tipo_pagina": tipo_pagina(r.url), "status": r.status,
            "fecha_captura": dt.datetime.now().isoformat(timespec="seconds"),
            "html": html, "title": r.css("title::text").get(),
            "meta_description": r.css('meta[name="description"]::attr(content)').get(),
            "h1": [t.strip() for t in r.css("h1 ::text").getall() if t.strip()][:3],
            **metadatos(html, r.url), "pdf_candidatos": pdfs,
        }
        if r.meta.get("es_extra"):
            return
        for href, texto in enlaces:
            destino = r.urljoin(href)
            if (urlparse(destino).netloc == urlparse(r.url).netloc and not destino.lower().endswith(".pdf")
                    and INSTITUCIONAL.search(href + " " + texto)
                    and self.extra_por_empresa.get(exp, 0) < PAGINAS_EXTRA):
                self.extra_por_empresa[exp] = self.extra_por_empresa.get(exp, 0) + 1
                yield r.follow(destino, meta={"es_extra": True})

## 3. Captura y escritura en bronce

In [5]:
inicio = time.time()
paginas = 0
with MANIFEST.open("a", encoding="utf-8") as manifest:
    async for item in CapturaBloque2(objetivo).stream():
        html = item.pop("html")
        huella = hashlib.sha256(html.encode("utf-8")).hexdigest()
        (WEB / f"{item['expediente']}_{huella}.html").write_text(html, encoding="utf-8")
        manifest.write(json.dumps({**item, "sha256": huella}, ensure_ascii=False) + "\n")
        paginas += 1
print(f"{paginas} páginas capturadas en {time.time() - inicio:.0f} s")

0 páginas capturadas en 0 s


## 4. Qué se obtuvo por empresa

In [6]:
registros = [json.loads(l) for l in MANIFEST.read_text(encoding="utf-8").splitlines() if l]
captura = pd.DataFrame(registros)
resumen = captura.groupby("expediente").agg(
    paginas=("url", "count"),
    tipos=("tipo_pagina", lambda s: ", ".join(sorted(set(s)))),
    con_og_description=("og_description", lambda s: s.notna().any()),
    con_meta_description=("meta_description", lambda s: s.notna().any()),
    con_jsonld=("jsonld_organizacion", lambda s: any(bool(x) for x in s)),
    pdf_candidatos=("pdf_candidatos", lambda s: len({p for lista in s for p in lista})),
).reset_index()
resumen.merge(urls[["expediente", "nombre"]], on="expediente")

,expediente,paginas,tipos,con_og_description,con_meta_description,con_jsonld,pdf_candidatos,nombre
0,384,4,"nosotros, otra, portada, servicios",True,True,True,0,CORPORACION FAVORITA C.A.
1,7154,3,"nosotros, portada",True,True,False,1,PROCESADORA NACIONAL DE ALIMENTOS C.A. PRONACA
2,124193,3,"otra, portada, servicios",True,True,False,0,MINUTOCORP S.A.
3,724114,4,"nosotros, otra, portada, servicios",False,False,False,0,FERTISA AGIF C.L.


In [7]:
captura[["expediente", "tipo_pagina", "title", "meta_description", "og_description"]].head(20)

,expediente,tipo_pagina,title,meta_description,og_description
0,384,portada,Inicio,"Corporación Favorita, empresa ecuatoriana mult...","Corporación Favorita, empresa ecuatoriana mult..."
1,7154,portada,\n Inicio,Somos una empresa multinacional familiar que a...,Somos una empresa multinacional familiar que a...
2,124193,portada,Minutocorp | Construyendo la Nueva Era | Promo...,Minutocorp es una constructora y promotora inm...,Minutocorp es una constructora y promotora inm...
3,384,nosotros,Quiénes somos | Corporación Favorita,"Conoce la visión, misión, historia y valores d...","Conoce la visión, misión, historia y valores d..."
4,724114,portada,Fertisa,NaN,NaN
5,384,servicios,Hanaska Servicios | Corporación Favorita,"Creada en 1973, presta servicios de alimentaci...","Creada en 1973, presta servicios de alimentaci..."
6,7154,nosotros,\n Acerca de nosotros,"Conoce sobre la historia, cultura y propósito ...","Conoce sobre la historia, cultura y propósito ..."
7,384,otra,Marcas | Corporación Favorita,Conoce todas las marcas de Corporación Favorit...,Conoce todas las marcas de Corporación Favorit...
8,7154,nosotros,\n Nuestra historia,Explora el legado de Pronaca y cómo hemos evol...,Explora el legado de Pronaca y cómo hemos evol...
9,124193,otra,Nuestro ADN – Minutocorp,ConstruyendoLa Nueva Era Somos una constructor...,ConstruyendoLa Nueva Era Somos una constructor...
